# Week 5: Classical ML Algorithms — Quick Reference

Cheat sheet and quick lookup for all algorithms covered in Week 5.

## 1. Decision Trees

### Key Concept
Recursively split data on features that minimize Gini impurity or MSE.

### Hyperparameters
- `max_depth`: Tree height (3–7 typical)
- `min_samples_split`: Min samples to split node (default 2)
- `min_samples_leaf`: Min samples in leaf node (default 1)
- `criterion`: 'gini' or 'entropy' (classification)

### When to Use
✓ Interpretability is critical
✓ Mixed feature types (categorical + numeric)
✗ Prone to overfitting
✗ Unstable (small data changes → large tree changes)

### Code Snippet
```python
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier(max_depth=5, min_samples_leaf=5)
dt.fit(X_train, y_train)
importance = dt.feature_importances_
```

## 2. Random Forests

### Key Concept
Bagging + random features per split. Average predictions from many decorrelated trees.

### Hyperparameters
- `n_estimators`: Number of trees (50–200 typical, >100 has diminishing returns)
- `max_depth`: Per-tree depth (often leave None or deep)
- `min_samples_leaf`: Min in leaf (default 1)
- `max_features`: Features per split ('sqrt' for classification)
- `oob_score`: Use OOB for free validation

### When to Use
✓ Fast training & prediction
✓ Little tuning needed
✓ Parallel-friendly
✓ Feature importance ranking
✗ Black box (less interpretable)

### Code Snippet
```python
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=100, oob_score=True, n_jobs=-1)
rf.fit(X_train, y_train)
oob_acc = rf.oob_score_
```

## 3. XGBoost

### Key Concept
Sequential boosting: each tree fits residuals of previous ensemble. Wins competitions.

### Hyperparameters
- `n_estimators`: Number of rounds (100–1000 typical, use early stopping)
- `learning_rate` (eta): 0.01–0.3 (lower = better, slower)
- `max_depth`: Per-tree depth (3–8 typical, shallow trees work best)
- `subsample`: Row sampling (0–1, default 1)
- `colsample_bytree`: Feature sampling (0–1, default 1)
- `lambda` (reg_lambda): L2 regularisation
- `alpha` (reg_alpha): L1 regularisation

### When to Use
✓ Best accuracy on tabular data
✓ Handles missing values natively
✓ Early stopping prevents overfitting
✓ Fast inference
✗ Slower training than RF
✗ Black box

### Code Snippet
```python
import xgboost as xgb
clf = xgb.XGBClassifier(learning_rate=0.1, max_depth=5, n_estimators=100)
clf.fit(X_train, y_train, eval_set=[(X_val, y_val)], early_stopping_rounds=20)
```

## 4. Support Vector Machines (SVM)

### Key Concept
Find maximum-margin hyperplane. With kernels, handles non-linear patterns.

### Hyperparameters
- `C`: Regularisation (high C = strict, low C = loose). Usually 0.1–100.
- `kernel`: 'linear', 'rbf', 'poly'. Start with 'rbf'.
- `gamma`: RBF kernel width. 'scale' or 'auto' recommended.
- `degree`: Polynomial degree (for poly kernel).
- `probability`: Set True for predict_proba().

### When to Use
✓ Small-medium datasets (< 100k)
✓ Interpretable decision boundaries
✓ Works with few samples
✗ Slow on large data
✗ Sensitive to scaling
✗ Black box with non-linear kernels

### Code Snippet
```python
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
svm = SVC(kernel='rbf', C=1.0, gamma='scale')
svm.fit(X_train_scaled, y_train)
```

## 5. K-Nearest Neighbors (KNN)

### Key Concept
Lazy learner: find K nearest training points, vote/average their labels.

### Hyperparameters
- `n_neighbors` (K): 3–10 typical. Tune via cross-validation.
- `metric`: 'euclidean', 'manhattan', 'cosine'. Distance metric.
- `weights`: 'uniform' or 'distance'. Distance-weighted = closer points count more.
- `algorithm`: 'auto', 'ball_tree', 'kd_tree', 'brute'.

### When to Use
✓ Simple baseline
✓ Non-parametric (no assumptions)
✗ Curse of dimensionality (use PCA first)
✗ Slow prediction on large data
✗ Sensitive to scaling

### Code Snippet
```python
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
knn = KNeighborsClassifier(n_neighbors=5, metric='euclidean')
knn.fit(X_train_scaled, y_train)
```

## Algorithm Comparison Matrix

| Aspect | DT | RF | XGBoost | SVM | KNN |
|--------|----|----|---------|-----|-----|
| Speed (Train) | Fast | Fast | Slow | Slow | Instant (lazy) |
| Speed (Pred) | Fast | Fast | Fast | Slow | Slow |
| Accuracy | Medium | High | **Best** | Medium | Medium |
| Interpretability | **Best** | Good | Poor | Good | Good |
| Scaling needed? | No | No | No | **Yes** | **Yes** |
| Missing values | Handles | Handles | **Handles** | Impute | Impute |
| Tuning effort | Low | Low | Medium | Medium | Low |
| Data types | Both | Both | Numeric | Numeric | Numeric |
| Large datasets | Good | **Good** | **Good** | Poor | Poor |

## Quick Decision Tree

1. **Is accuracy your only goal?** → Use XGBoost
2. **Need interpretability?** → Use Decision Tree
3. **Large dataset, need speed?** → Use Random Forest
4. **Small dataset, want to try kernels?** → Use SVM (RBF)
5. **Quick baseline?** → Use KNN or Logistic Regression

## Hyperparameter Tuning Best Practices

### GridSearchCV vs RandomizedSearchCV
```python
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

# GridSearchCV: exhaustive search (small parameter space)
params = {'max_depth': [3, 5, 7], 'n_estimators': [50, 100, 200]}
grid = GridSearchCV(RandomForestClassifier(), params, cv=5)

# RandomizedSearchCV: random sample (large parameter space)
distributions = {'max_depth': [3, 5, 7, 10, 15], 'learning_rate': [0.01, 0.05, 0.1, 0.2]}
random = RandomizedSearchCV(XGBClassifier(), distributions, n_iter=20, cv=5)
```

### Cross-Validation
```python
from sklearn.model_selection import StratifiedKFold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# Always use StratifiedKFold for classification (preserves class distribution)
```

### Early Stopping (XGBoost)
```python
clf.fit(X_train, y_train, eval_set=[(X_val, y_val)],
        early_stopping_rounds=20, verbose=False)
```

## Common Mistakes & How to Avoid Them

1. **Forgetting to scale SVM/KNN inputs**
   → Always: `StandardScaler().fit_transform(X_train)`

2. **Tuning hyperparameters on test set**
   → Proper split: train (60%), validation (20%), test (20%)

3. **Not checking for data leakage**
   → Fit scaler/imputer ONLY on train set, apply to val/test

4. **Using accuracy on imbalanced data**
   → Use F1, precision, recall, or AUC instead

5. **XGBoost without early stopping**
   → Always use `early_stopping_rounds` to prevent overfitting

6. **Not checking feature importance**
   → Trees give you free feature importance—use it for debugging

7. **Ignoring OOB score in Random Forests**
   → Set `oob_score=True`, free CV with zero overhead

## Production Checklist

Before deploying any model:

- [ ] Train/validation/test split (no data leakage)
- [ ] Scaling/preprocessing fit on train only
- [ ] Cross-validation on train+val
- [ ] Final evaluation on held-out test set
- [ ] Confidence intervals on metrics (bootstrap)
- [ ] Check for class imbalance → use stratified CV
- [ ] Feature importance analysis (debugging)
- [ ] Model card: what it does, limitations, bias
- [ ] Latency & throughput benchmarks
- [ ] Monitoring: performance degradation over time
- [ ] Versioning: model + preprocessing pipeline + config